In [ ]:
!nvidia-smi

Thu Feb 26 19:05:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [5]:
import re
import time
import json
import os
import numpy as np
from datasets import load_dataset
from tqdm import tqdm

# Load GSM8K test set
print('Loading GSM8K test set...')
ds = load_dataset('gsm8k', 'main', split='test')

def extract_answer_number(text):
    """
    Extracts the final numeric answer from GSM8K-style text.
    Looks for the '####' marker first, then falls back to the last number.
    """
    if '####' in text:
        text = text.split('####')[-1]
    text = text.replace(',', '')
    matches = re.findall(r'-?\d+\.?\d*', text)
    return matches[-1] if matches else None

# Set eval size — use len(ds) for full run, or 50 for a quick smoke test
EVAL_SIZE = len(ds)
# EVAL_SIZE = 50

print(f'Dataset loaded. Evaluating on {EVAL_SIZE} examples.')


Loading GSM8K test set...


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Dataset loaded. Evaluating on 1319 examples.


In [6]:
# Install vLLM (includes torch, transformers, etc. — no separate installs needed)
# This replaces the previous torch + transformers + bitsandbytes install
!pip install vllm lm-format-enforcer pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 508.3/508.3 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.6/192.6 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 90.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 64.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 74.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 M

In [ ]:
from IPython.display import display, Markdown

def display_header(text):
    display(Markdown(f'**{text}**'))

def display_content(text):
    display(Markdown(f'```\n{text}\n```'))


In [ ]:
# --- Config ---
MODEL_NAME = "Qwen/Qwen3-8B"
BATCH_SIZE = 4          # Increased since 4-bit uses less VRAM
MAX_NEW_TOKENS = 512    # Kept at 512 for safety on harder reasoning, but can lower to 256
EVAL_SIZE = len(ds)     # Run full test set
ENABLE_THINKING = True

In [ ]:
# Fix Protobuf compatibility issue
!pip install "protobuf==3.20.3"
# After running this cell, please RESTART THE SESSION (Runtime -> Restart session)
# and then run the notebook again.

In [ ]:
if __name__ == '__main__':
  from vllm import LLM, SamplingParams

  print('Loading Qwen3-8B with vLLM...')
  print( '  PagedAttention: enabled (automatic)')
  print( '  Cont. batching: enabled (automatic)')
  print( '  dtype=half    : required for Colab T4 (no bfloat16 on compute cap 7.5)')

  llm = LLM(
      model=MODEL_NAME,
      dtype='float16',                  # float16 — required for T4; use 'bfloat16' on A100
      gpu_memory_utilization=0.95,   # Increased slightly to squeeze in weights
      max_model_len=4096,            # Reduced to 4096 to prevent OOM on T4 (16GB is tight for 8B params)
      enforce_eager=False,
  )

  # Qwen3 recommended sampling params:
  sampling_params = SamplingParams(
      temperature=0,          # greedy
      top_k=20,
      max_tokens=MAX_NEW_TOKENS,
  )

  print('Model loaded successfully!')

Loading Qwen3-8B with vLLM...
  PagedAttention: enabled (automatic)
  Cont. batching: enabled (automatic)
  dtype=half    : required for Colab T4 (no bfloat16 on compute cap 7.5)
INFO 03-03 04:39:21 [utils.py:223] non-default args: {'dtype': 'float16', 'max_model_len': 4096, 'gpu_memory_utilization': 0.95, 'disable_log_stats': True, 'model': 'Qwen/Qwen3-8B'}
INFO 03-03 04:39:21 [model.py:529] Resolved architecture: Qwen3ForCausalLM
WARNING 03-03 04:39:21 [model.py:1874] Casting torch.bfloat16 to torch.float16.
INFO 03-03 04:39:21 [model.py:1549] Using max model len 4096
INFO 03-03 04:39:21 [scheduler.py:224] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 03-03 04:40:25 [llm.py:355] Supported tasks: ['generate']
Model loaded successfully!


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

DRIVE_SAVE_DIR = '/content/drive/MyDrive/Colab_Data/Qwen3_8B_GSM8K_Eval_vLLM_thinking'
import os
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

print(f'Saving results to: {DRIVE_SAVE_DIR}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saving results to: /content/drive/MyDrive/Colab_Data/Qwen3_8B_GSM8K_Eval_vLLM_thinking


In [ ]:
# --- Checkpoint Setup ---
# vLLM generates everything in one call, so checkpointing works differently here:
# we checkpoint the RESULTS after generation, not during it.
# If the session dies mid-generation, re-run this cell — it will skip to post-processing.
CHECKPOINT_FILE = os.path.join(DRIVE_SAVE_DIR, 'qwen3_8b_gsm8k_vllm_thinking_checkpoint.jsonl')
OUTPUTS_CACHE   = os.path.join(DRIVE_SAVE_DIR, 'qwen3_8b_gsm8k_thinking_vllm_raw_outputs.json')

results = []
if os.path.exists(CHECKPOINT_FILE):
    print(f'Found checkpoint: {CHECKPOINT_FILE}')
    with open(CHECKPOINT_FILE) as f:
        for line in f:
            try:
                results.append(json.loads(line))
            except json.JSONDecodeError:
                continue
    print(f'Loaded {len(results)} previously evaluated examples.')
else:
    print('No checkpoint found — starting from scratch.')

# Prepare the full question + ground-truth lists
subset        = ds.select(range(EVAL_SIZE))
questions     = subset['question']
ground_truths = [extract_answer_number(a) for a in subset['answer']]

# Determine which examples still need to be generated
already_done = len(results)
remaining_qs = questions[already_done:]
print(f'{already_done} already done, {len(remaining_qs)} remaining.')


No checkpoint found — starting from scratch.
0 already done, 1319 remaining.


In [3]:
# --- Build Prompts ---
# Qwen3-specific: apply_chat_template accepts an enable_thinking kwarg.
#
#   enable_thinking=True  (default) — inserts a special token that tells the model
#                                     to produce a <think>...</think> reasoning block
#                                     before the final answer. Higher accuracy, slower.
#
#   enable_thinking=False           — model answers directly with no CoT preamble.
#                                     Faster and cheaper, somewhat lower accuracy.
#
# This is controlled by the ENABLE_THINKING flag set in the Config cell.
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

prompts = [
    tokenizer.apply_chat_template(
        [{'role': 'user', 'content': q}],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=ENABLE_THINKING,  # <-- Qwen3-specific kwarg
    )
    for q in remaining_qs
]

print(f'Built {len(prompts)} prompts (enable_thinking={ENABLE_THINKING}).')
print('\nExample prompt (first 400 chars):')
print(prompts[0][:400] if prompts else '(none — all done)')


KeyboardInterrupt: 

In [2]:
# --- Generate with vLLM ---
# Key difference from the original HuggingFace loop:
#
#   BEFORE (HuggingFace):  manual for-loop, one batch of 4 at a time,
#                          each batch blocks until ALL sequences in it finish.
#
#   NOW (vLLM):            pass ALL prompts at once, vLLM's continuous
#                          batching scheduler handles everything internally —
#                          short responses free their slots immediately so
#                          longer responses keep the GPU fully utilised.

if prompts:
    print(f'Generating responses for {len(prompts)} prompts...')
    print('(vLLM will log per-step throughput below)')

    gen_start = time.time()
    outputs = llm.generate(prompts, sampling_params)  # single call — no loop needed
    gen_time = time.time() - gen_start

    total_new_tokens = sum(len(o.outputs[0].token_ids) for o in outputs)
    throughput = total_new_tokens / gen_time

    print(f'\nGeneration complete.')
    print(f'  Time:       {gen_time/60:.1f} min')
    print(f'  Tokens:     {total_new_tokens:,}')
    print(f'  Throughput: {throughput:.1f} tokens/sec')

    # Cache raw outputs to Drive so we can re-run post-processing without re-generating
    raw_cache = [
        {
            'question_idx': already_done + i,
            'response': o.outputs[0].text,
            'n_tokens': len(o.outputs[0].token_ids),
        }
        for i, o in enumerate(outputs)
    ]
    with open(OUTPUTS_CACHE, 'w') as f:
        json.dump(raw_cache, f)
    print(f'  Raw outputs cached to: {OUTPUTS_CACHE}')
else:
    print('No prompts to generate — loading cached outputs.')
    with open(OUTPUTS_CACHE) as f:
        raw_cache = json.load(f)
    gen_time = None
    throughput = None
    total_new_tokens = sum(r['n_tokens'] for r in raw_cache)


NameError: name 'prompts' is not defined

In [1]:
# --- Score Results & Save Checkpoint ---
#
# Qwen3 in thinking mode produces output like:
#   <think>\n...reasoning steps...\n</think>\n42
# We must strip the <think> block before running extract_answer_number,
# otherwise the regex may latch onto numbers inside the reasoning trace
# rather than the final answer.

def strip_thinking(text):
    """Remove <think>...</think> block from Qwen3 output, return only the final answer part."""
    if '</think>' in text:
        return text.split('</think>', 1)[-1].strip()
    return text

new_results = []
for item in raw_cache:
    idx      = item['question_idx']
    response = item['response']

    # Strip CoT block before extracting the numeric answer
    answer_text = strip_thinking(response) if ENABLE_THINKING else response
    pred = extract_answer_number(answer_text)
    gt   = ground_truths[idx]

    is_correct = False
    if pred is not None and gt is not None:
        try:
            is_correct = float(pred) == float(gt)
        except ValueError:
            pass

    res = {
        'question_idx': idx,
        'prediction':   pred,
        'ground_truth': gt,
        'is_correct':   is_correct,
        'is_unknown':   (pred is None),
        'new_tokens':   item['n_tokens'],
    }
    new_results.append(res)
    results.append(res)

# Append new results to checkpoint file
with open(CHECKPOINT_FILE, 'a') as f:
    for r in new_results:
        f.write(json.dumps(r) + '\n')

# --- Final Metrics ---
correct_count  = sum(1 for r in results if r['is_correct'])
unknown_count  = sum(1 for r in results if r['is_unknown'])
all_new_tokens = sum(r['new_tokens'] for r in results)

final_metrics = {
    'method':             f'icl_0_shot_vllm_thinking={ENABLE_THINKING}',
    'model':              MODEL_NAME,
    'eval_size':          len(results),
    'accuracy':           correct_count / len(results) if results else 0,
    'unknown_frac':       unknown_count / len(results) if results else 0,
    'total_new_tokens':   all_new_tokens,
    'gen_tokens_per_sec': throughput if throughput else 'N/A (loaded from cache)',
    'total_gen_time_min': gen_time / 60 if gen_time else 'N/A (loaded from cache)',
}

print('\n=== Final Metrics ===')
for k, v in final_metrics.items():
    print(f'  {k}: {v}')

# Save final metrics summary to Drive
METRICS_FILE = os.path.join(DRIVE_SAVE_DIR, 'final_metrics.json')
with open(METRICS_FILE, 'w') as f:
    json.dump(final_metrics, f, indent=2)
print(f'\nMetrics saved to: {METRICS_FILE}')


NameError: name 'raw_cache' is not defined